# 국가별 안전정보 등록 건수 분석

## 1. 분석 목적

연도별 안전정보 등록 현황을 확인하고, 각 연도에서 안전정보가 많이 등록된 국가 TOP10을 비교하고자 한다.

이를 통해 안전정보가 특정 연도나 국가에 집중되어 등록되는 경향이 있는지 확인한다.

단, 안전정보 등록 건수는 실제 국가의 위험도를 직접적으로 나타내는 지표가 아니므로, 본 분석에서는 국가별 위험도를 평가하기보다 안전정보 등록 현황을 파악하는 데 초점을 둔다.

## 2. 데이터 불러오기 및 기본 정보 확인

분석에 사용할 `safety_stats.csv` 데이터를 불러온 후 데이터의 크기, 컬럼 구성, 연도 범위를 확인한다.

In [1]:
import pandas as pd
import plotly.express as px

safety_info_stats = pd.read_csv(
    "../data/safety_stats.csv"
)

In [2]:
display(safety_info_stats.head())
print(safety_info_stats.shape)
print(safety_info_stats.columns)

print("컬럼:", safety_info_stats.columns.tolist())
print("연도 목록:")
print(sorted(safety_info_stats["year"].unique()))

print("연도 개수:", safety_info_stats["year"].nunique())

,iso3,country_kr,year,count
0,AFG,아프가니스탄,2011,1
1,AFG,아프가니스탄,2012,3
2,AFG,아프가니스탄,2013,3
3,AFG,아프가니스탄,2014,18
4,AFG,아프가니스탄,2015,42


(1134, 4)
Index(['iso3', 'country_kr', 'year', 'count'], dtype='str')
컬럼: ['iso3', 'country_kr', 'year', 'count']
연도 목록:
[np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
연도 개수: 15


### 데이터 확인 결과

데이터에는 연도별·국가별 안전정보 등록 건수가 포함되어 있으며, `year`를 기준으로 연도별 현황을 비교하고 `count`를 기준으로 국가별 등록 건수를 비교할 수 있다.

이후 분석에서는 연도별 국가 안전정보 등록 건수와 TOP10 국가의 분포를 중심으로 살펴본다.

## 3. 2024년 국가별 안전정보 등록 건수 TOP10

특정 연도 내에서 국가별 안전정보 등록 건수에 어떤 차이가 있는지 확인하기 위해 2024년 데이터를 기준으로 등록 건수가 많은 상위 10개 국가를 추출한다.

In [3]:
selected_year = 2024

top10 = (
    safety_info_stats[safety_info_stats["year"] == selected_year]
    .sort_values("count", ascending=False)
    .head(10)
)

display(top10)

,iso3,country_kr,year,count
157,CHL,칠레,2024,3
169,CHN,중국,2024,2
256,ECU,에콰도르,2024,2
218,DEU,독일,2024,2
781,PNG,파푸아뉴기니,2024,2
44,AUS,호주,2024,1
83,BGD,방글라데시,2024,1
116,BOL,볼리비아,2024,1
195,CUB,쿠바,2024,1
33,ARG,아르헨티나,2024,1


In [4]:
fig = px.bar(
    top10,
    x="count",
    y="country_kr",
    orientation="h",
    title=f"{selected_year}년 국가별 안전정보 건수 TOP 10",
    labels={
        "count": "안전정보 건수",
        "country_kr": "국가"
    },
    text="count"
)

# 안전정보가 많은 국가가 위쪽에 오도록 정렬
fig.update_layout(
    yaxis={
        "categoryorder": "total ascending"
    },
    xaxis={
        "dtick": 1
    }
)

fig.show()

### 분석 결과

2024년 데이터를 기준으로 국가별 안전정보 등록 건수를 비교한 결과, 상위 국가 간에도 등록 건수에 차이가 있음을 확인할 수 있다.

다만 2024년 데이터는 7월 23일까지 수록된 자료이므로 완전한 연간 데이터가 아니다. 따라서 다른 연도의 전체 등록 건수와 직접 비교하기보다는 2024년 내에서 국가별 등록 현황을 비교하는 용도로 해석한다.

## 4. 2011~2024년 연도별 국가 안전정보 TOP10

특정 연도뿐만 아니라 연도에 따라 안전정보 등록 현황이 어떻게 달라지는지 확인하기 위해 2011년부터 2024년까지 각 연도의 국가별 안전정보 등록 건수 TOP10을 비교한다.

각 연도별 전체 안전정보 등록 건수와 TOP10 국가의 등록 건수 합계를 함께 확인하여 연도별 데이터 규모의 차이도 살펴본다.

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

years = sorted(safety_info_stats["year"].unique())
years = [year for year in years if year >= 2011]

# 각 연도별 제목 만들기
subplot_titles = []

for year in years:
    year_data = safety_info_stats[
        safety_info_stats["year"] == year
    ]

    # 해당 연도 전체 안전정보 건수
    total_count = year_data["count"].sum()

    # 해당 연도 TOP 10 안전정보 건수
    top10_count = (
        year_data
        .nlargest(10, "count")["count"]
        .sum()
    )

    subplot_titles.append(
        f"{year}년 | 전체 {total_count}건 · TOP10 {top10_count}건"
    )


fig = make_subplots(
    rows=5,
    cols=3,
    subplot_titles=subplot_titles,
    vertical_spacing=0.06,
    horizontal_spacing=0.08
)


for i, year in enumerate(years):
    row = i // 3 + 1
    col = i % 3 + 1

    top10 = (
        safety_info_stats[safety_info_stats["year"] == year]
        .sort_values("count", ascending=False)
        .head(10)
        .sort_values("count", ascending=True)
    )

    fig.add_trace(
        go.Bar(
            x=top10["count"],
            y=top10["country_kr"],
            orientation="h",
            text=top10["count"],
            textposition="outside",
            showlegend=False
        ),
        row=row,
        col=col
    )

fig.update_layout(
    height=1800,
    width=1200,
    title="2011~2024년 연도별 국가 안전정보 등록 건수 TOP 10",
    showlegend=False
)

fig.show()

### 분석 결과

연도별 안전정보 등록 건수를 비교한 결과, 연도에 따라 전체 등록 건수의 편차가 크게 나타났다.

특히 일부 연도에는 안전정보 등록이 상대적으로 집중되어 있었으며, 2021년 이후에는 이전 시기에 비해 전체 등록 건수가 크게 감소하는 모습을 확인할 수 있다.

또한 각 연도의 TOP10 국가 구성이 동일하지 않아 시기에 따라 안전정보가 집중된 국가에도 차이가 있음을 확인하였다.

그러나 현재 데이터만으로 연도별 등록 건수 변화의 원인을 특정하기는 어렵다. 특정 사건의 발생, 안전공지 등록 정책, 데이터 수집 범위 등의 영향을 받을 수 있기 때문이다.

따라서 연도별 안전정보 등록 건수의 차이를 실제 국가 위험도의 증가 또는 감소로 해석하지 않는다.

## 5. 시각화 결과 저장

분석 결과를 대시보드 및 프로젝트 산출물에서 활용할 수 있도록 연도별 국가 안전정보 TOP10 그래프를 이미지 파일로 저장한다.

In [ ]:
# 폴더 생성 후 이미지 저장

from pathlib import Path

Path("../outputs/charts").mkdir(parents=True, exist_ok=True)

fig.write_image(
    "../outputs/charts/safety_info_top10_by_year.png",
    width=1200,
    height=1800,
    scale=2
)

## 6. 결론 및 분석 한계

이번 분석을 통해 연도별 안전정보 등록 건수의 편차가 크고, 각 연도에서 안전정보가 많이 등록된 국가의 구성에도 차이가 있음을 확인하였다.

특히 2021년 이후 등록 건수가 이전보다 감소하는 경향이 나타났지만, 현재 데이터만으로 그 원인을 특정하기는 어렵다.

또한 안전정보 등록 건수는 실제 국가의 위험 수준을 직접적으로 나타내는 지표가 아니다. 특정 사건 발생, 정보 등록 정책, 데이터 수집 범위 등에 따라 등록 건수가 달라질 수 있기 때문이다.

2024년 데이터는 7월 23일까지 수록되어 있어 완전한 연간 데이터가 아니므로 다른 연도와의 직접적인 절대 건수 비교에도 주의가 필요하다.

따라서 본 분석은 국가의 위험도를 평가하기보다, 연도별 안전정보 등록 현황과 각 연도 내 국가별 TOP10 분포를 파악하는 데 의미가 있다.